In [2]:
import sys
!{sys.executable} -m pip install hdfs


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install delta-spark

^C
Note: you may need to restart the kernel to use updated packages.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import *

# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/AVI'

builder = SparkSession \
    .builder \
    .appName("Python Spark SQL Hive integration example") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:1.0.0") \
    .enableHiveSupport() \

spark = spark = configure_spark_with_delta_pip(builder).getOrCreate()

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
# Importando as bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, split, unix_timestamp, expr

# Criando a sessão Spark
spark = SparkSession.builder \
    .appName("Manipulação de Dados com PySpark e Delta Lake") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Lendo o arquivo CSV local
call = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("caminho/do/seu/arquivo/911.csv")

# Renomeando a coluna 'twp' para 'town'
call = call.drop("e").withColumnRenamed("twp", "town")

# Separando colunas de data e hora da coluna 'timeStamp'
call = call.withColumn('arrival_date', split(call['timeStamp'], ' ').getItem(0))
call = call.withColumn('arrival_time', split(call['timeStamp'], ' ').getItem(1))
call = call.drop('timeStamp')

# Expressão regular para extrair data e hora da coluna 'desc'
date_expr = r'(\d{4}-\d{2}-\d{2}) @ (\d{2}:\d{2}:\d{2})|(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})'
call = call.withColumn('data_hora', split(call['desc'], date_expr, 0))
call = call.withColumn('call_date', when(call['data_hora'] != "", split(call['data_hora'], r'(\d{4}-\d{2}-\d{2})', 0)).otherwise(None))
call = call.withColumn('call_time', when(call['data_hora'] != "", split(call['data_hora'], r'(\d{2}:\d{2}:\d{2})', 0)).otherwise(None))
call = call.drop('data_hora')

# Tratando valores nulos e realizando outras operações necessárias

# Calculando 'reply_time' com base em 'call_date' e 'call_time' versus 'arrival_date' e 'arrival_time'
call = call \
    .withColumn("reply_time", 
        when(col("arrival_date") == col("call_date"),
            (unix_timestamp("arrival_time", "HH:mm:ss") - unix_timestamp("call_time", "HH:mm:ss"))
        )
        .otherwise(
            (24 * 3600) - (unix_timestamp("arrival_time", "HH:mm:ss") - unix_timestamp("call_time", "HH:mm:ss"))
        )
    ) \
    .dropDuplicates() \
    .withColumn("reply_time", col("reply_time").cast("integer"))

# Escrevendo os dados em uma tabela Delta
call \
    .select("lat", "lng", "zip", "title", "town", "arrival_date", "arrival_time", "call_date", "call_time", "type", "year", "reply_time") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("year") \
    .option("overwriteSchema", "true") \
    .save("caminho/do/seu/delta/table")

# Verificando os resultados na tabela Delta criada
call_delta = spark.read.format("delta").load("caminho/do/seu/delta/table")
call_delta.show()
call_delta.count()


ModuleNotFoundError: No module named 'pyspark'